# ENARES 2024 — Construcción de indicadores según SPSS

Este cuaderno reconstruye los indicadores en `analytical_crs04_adolescents`. La autoridad son las sintaxis `.sps`; los notebooks anteriores y sus diccionarios no se usan como fuente lógica.

Ejecuta las celdas en orden. Cada bloque reemplaza únicamente sus columnas derivadas y conserva las demás columnas y las 18,807 filas.


## 0. Conexión y comprobación de la tabla


In [1]:
!pip install -q google-cloud-bigquery pandas pandas-gbq pyarrow db-dtypes openpyxl XlsxWriter tabulate
from google.colab import auth, drive
from google.cloud import bigquery
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd, hashlib

auth.authenticate_user()
drive.mount('/content/drive')

PROJECT_ID = 'enares-2024-crs04'
LOCATION = 'US'
EXPECTED_ROWS = 18807
ROOT_DRIVE = Path('/content/drive/MyDrive/ENARES_2024_PROJECT')
LOG_DIR = ROOT_DRIVE / '05Resultados' / 'logs' / 'stage03'
SQL_DIR = ROOT_DRIVE / '02SQL'
for folder in [LOG_DIR, SQL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_UTC = datetime.now(timezone.utc).isoformat()
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
A = f'{PROJECT_ID}.enares2024_crs04_analytical.analytical_crs04_adolescents'
SPSS_MATERIALIZED = set()
SPSS_BLOCK_AUDIT = []

table = client.get_table(A)
if table.num_rows != EXPECTED_ROWS:
    raise RuntimeError(f'{A}: {table.num_rows} filas; se esperaban {EXPECTED_ROWS}')
print('Tabla analítica:', A)
print('Filas:', table.num_rows, '| ejecución UTC:', RUN_UTC)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 3.1 MB/s eta 0:00:00
Mounted at /content/drive
Tabla analítica: enares-2024-crs04.enares2024_crs04_analytical.analytical_crs04_adolescents
Filas: 18807 | ejecución UTC: 2026-08-16T19:21:44.078644+00:00


## 1. Controles previos de Stage 03

Se valida el universo, la llave disponible, tipos FLOAT64, variables críticas y nulos. Este flujo comienza en la tabla analítica ya existente; no vuelve a unir las tablas raw/cleaned.


In [2]:
schema_fields={field.name:field.field_type for field in client.get_table(A).schema}
contract_vars=['ID','COLEGIAL_ID','FACTOR_ALUMNOS','CCDD','SEXO','AREA']
contract=pd.DataFrame({
    'variable':contract_vars,
    'present_in_analytical':[name in schema_fields for name in contract_vars],
    'type':[schema_fields.get(name) for name in contract_vars],
    'required_for_run':[name in {'ID','FACTOR_ALUMNOS','CCDD'} for name in contract_vars]})
contract.to_csv(LOG_DIR/'stage3_variable_contract_check.csv',index=False)
blocking=contract.loc[contract.required_for_run & ~contract.present_in_analytical,'variable'].tolist()
if blocking:
    raise RuntimeError(f'Contrato mínimo incompleto: {blocking}')

# La guía propone ID + COLEGIAL_ID. Si COLEGIAL_ID está disponible se usa la
# llave compuesta; de lo contrario ID debe ser único para continuar.
key_cols=['ID']+(['COLEGIAL_ID'] if 'COLEGIAL_ID' in schema_fields else [])
key_select=', '.join(f'`{x}`' for x in key_cols)
key_check=client.query(f'''WITH row_counts AS (
  SELECT COUNT(*) AS total_rows,
         COUNTIF({' OR '.join(f'`{x}` IS NULL' for x in key_cols)}) AS null_key_rows
  FROM `{A}`
), key_counts AS (
  SELECT COUNT(*) AS distinct_keys
  FROM (SELECT {key_select} FROM `{A}` GROUP BY {key_select})
)
SELECT total_rows, null_key_rows, distinct_keys,
       total_rows-distinct_keys AS duplicated_key_rows
FROM row_counts CROSS JOIN key_counts''').result().to_dataframe()
key_check['key_used']='+'.join(key_cols)
key_check.to_csv(LOG_DIR/'stage3_key_validation.csv',index=False)
row=key_check.iloc[0]
if row.total_rows!=EXPECTED_ROWS or row.null_key_rows or row.duplicated_key_rows:
    raise RuntimeError('La llave o el universo analítico no aprobaron el control previo.')

float_checks=[]
for name in key_cols:
    if schema_fields[name]=='FLOAT64':
        bad=client.query(f'''SELECT COUNTIF(`{name}` IS NOT NULL AND `{name}`!=FLOOR(`{name}`)) AS n
                            FROM `{A}`''').result().to_dataframe().iloc[0,0]
        float_checks.append({'variable':name,'type':'FLOAT64','rows_with_decimals':int(bad)})
pd.DataFrame(float_checks,columns=['variable','type','rows_with_decimals']).to_csv(
    LOG_DIR/'stage3_float_key_check.csv',index=False)
if any(x['rows_with_decimals'] for x in float_checks):
    raise RuntimeError('La llave FLOAT64 contiene decimales reales.')

critical=[x for x in ['ID','COLEGIAL_ID','FACTOR_ALUMNOS','CCDD','SEXO','AREA'] if x in schema_fields]
quality=[]
for variable in critical:
    q=client.query(f'''SELECT COUNT(*) AS total_rows, COUNTIF(`{variable}` IS NULL) AS null_rows
                       FROM `{A}`''').result().to_dataframe().iloc[0]
    quality.append({'variable':variable,'total_rows':int(q.total_rows),'null_rows':int(q.null_rows),
                    'null_rate':float(q.null_rows/q.total_rows)})
quality=pd.DataFrame(quality)
quality.to_csv(LOG_DIR/'stage3_data_quality_report.csv',index=False)

display(contract)
display(key_check)
display(quality)
print('PASS preflight | llave:', '+'.join(key_cols))


,variable,present_in_analytical,type,required_for_run
0,ID,True,FLOAT,True
1,COLEGIAL_ID,True,FLOAT,False
2,FACTOR_ALUMNOS,True,FLOAT,True
3,CCDD,True,STRING,True
4,SEXO,True,FLOAT,False
5,AREA,True,FLOAT,False


,total_rows,null_key_rows,distinct_keys,duplicated_key_rows,key_used
0,18807,0,18807,0,ID+COLEGIAL_ID


,variable,total_rows,null_rows,null_rate
0,ID,18807,0,0.0
1,COLEGIAL_ID,18807,0,0.0
2,FACTOR_ALUMNOS,18807,0,0.0
3,CCDD,18807,0,0.0
4,SEXO,18807,0,0.0
5,AREA,18807,0,0.0


PASS preflight | llave: ID+COLEGIAL_ID


## 2. Mapa auditable de saltos y SYSMIS


In [3]:
# Registro documental de filtros observados en las sintaxis. La implementación
# efectiva permanece en los CASE de los bloques SPSS siguientes.
skip_map=pd.DataFrame([
 {'block':'VP_HOGAR','gateway_var':'C3P203','open_value':1,'dependent_vars':'C3P201_1..C3P201_11','recode':'SYSMIS->0 según SPSS','keep_null_when':'C3P203 IS NULL','source_module':'3.2'},
 {'block':'VF_HOGAR','gateway_var':'C3P207','open_value':1,'dependent_vars':'C3P205_1..C3P205_7','recode':'SYSMIS->0 según SPSS','keep_null_when':'C3P207 IS NULL','source_module':'3.2'},
 {'block':'VF_HOGAR_01','gateway_var':'C3P121','open_value':1,'dependent_vars':'C3P121','recode':'SYSMIS->0 según SPSS','keep_null_when':'','source_module':'3.2'},
 {'block':'VF_HOGAR_03','gateway_var':'C3P216A_*/C3P216C_*','open_value':1,'dependent_vars':'C3P216A_1..6;C3P216C_1..5;flags C','recode':'SYSMIS->0 según SPSS','keep_null_when':'','source_module':'3.2'},
 {'block':'DESP','gateway_var':'C3P105','open_value':'!=2','dependent_vars':'VFNNTV','recode':'conservar SYSMIS','keep_null_when':'C3P105=2','source_module':'3.2'},
 {'block':'VP_ESCUELA','gateway_var':'C3P225','open_value':1,'dependent_vars':'C3P223_1..C3P223_14','recode':'SYSMIS->0 según SPSS','keep_null_when':'C3P225 IS NULL','source_module':'3.3'},
 {'block':'VF_ESCUELA','gateway_var':'C3P229','open_value':1,'dependent_vars':'C3P227_1..C3P227_10','recode':'SYSMIS->0 según SPSS','keep_null_when':'C3P229 IS NULL','source_module':'3.3'},
 {'block':'VS_12M/VS_ESCUELA','gateway_var':'últimos 12 meses','open_value':1,'dependent_vars':'C4P248_1..C4P248_16','recode':'SYSMIS->0 según SPSS','keep_null_when':'','source_module':'3.4'},
 {'block':'ayuda_hogar_*','gateway_var':'C3P209/universo víctima hogar','open_value':1,'dependent_vars':'formas de búsqueda y recepción hogar','recode':'SYSMIS->0 dentro del dominio','keep_null_when':'fuera del dominio','source_module':'3.6'},
 {'block':'ayuda_escuela_*','gateway_var':'universo víctima escuela','open_value':1,'dependent_vars':'formas de búsqueda y recepción escuela','recode':'SYSMIS->0 dentro del dominio','keep_null_when':'fuera del dominio','source_module':'3.6'},
 {'block':'ayuda_vs_*','gateway_var':'universo víctima VS','open_value':1,'dependent_vars':'formas de búsqueda, institución y recepción VS','recode':'SYSMIS->0 dentro del dominio','keep_null_when':'fuera del dominio','source_module':'3.6'},
 {'block':'agresores_hogar','gateway_var':'forma hogar marcada','open_value':1,'dependent_vars':'grupos de agresor VP/VF hogar','recode':'SYSMIS->0 según SPSS','keep_null_when':'','source_module':'3.2.1'},
 {'block':'agresores_escuela','gateway_var':'forma escuela marcada','open_value':1,'dependent_vars':'grupos de agresor VP/VF escuela','recode':'SYSMIS->0 según SPSS','keep_null_when':'','source_module':'3.3.1'}])
skip_map.to_csv(LOG_DIR/'stage3_skip_map.csv',index=False)
display(skip_map)


,block,gateway_var,open_value,dependent_vars,recode,keep_null_when,source_module
0,VP_HOGAR,C3P203,1,C3P201_1..C3P201_11,SYSMIS->0 según SPSS,C3P203 IS NULL,3.2
1,VF_HOGAR,C3P207,1,C3P205_1..C3P205_7,SYSMIS->0 según SPSS,C3P207 IS NULL,3.2
2,VF_HOGAR_01,C3P121,1,C3P121,SYSMIS->0 según SPSS,,3.2
3,VF_HOGAR_03,C3P216A_*/C3P216C_*,1,C3P216A_1..6;C3P216C_1..5;flags C,SYSMIS->0 según SPSS,,3.2
4,DESP,C3P105,!=2,VFNNTV,conservar SYSMIS,C3P105=2,3.2
5,VP_ESCUELA,C3P225,1,C3P223_1..C3P223_14,SYSMIS->0 según SPSS,C3P225 IS NULL,3.3
6,VF_ESCUELA,C3P229,1,C3P227_1..C3P227_10,SYSMIS->0 según SPSS,C3P229 IS NULL,3.3
7,VS_12M/VS_ESCUELA,últimos 12 meses,1,C4P248_1..C4P248_16,SYSMIS->0 según SPSS,,3.4
8,ayuda_hogar_*,C3P209/universo víctima hogar,1,formas de búsqueda y recepción hogar,SYSMIS->0 dentro del dominio,fuera del dominio,3.6
9,ayuda_escuela_*,universo víctima escuela,1,formas de búsqueda y recepción escuela,SYSMIS->0 dentro del dominio,fuera del dominio,3.6


## 3. Función común para materializar un bloque SPSS


In [4]:
# ============================================================
# SPSS 3.1: implementación literal de variables derivadas y dimensiones
# Autoridad: 07_CRS04_3.1_Caracteristicas_violencia_Percepciones_ver6.sps
# No usa columnas derivadas por notebooks anteriores como fuente.
# ============================================================

A=globals().get('A',f'{PROJECT_ID}.enares2024_crs04_analytical.analytical_crs04_adolescents')

def apply_spss_block(block_name, required, expressions):
    """Reemplaza solo las columnas de salida del bloque, conservando la base."""
    schema={f.name for f in client.get_table(A).schema}
    missing=sorted(set(required)-schema)
    if missing:
        raise RuntimeError(f'{block_name}: faltan variables fuente SPSS: {missing}')
    replace=[name for name in expressions if name in schema]
    base='* EXCEPT('+','.join(f'`{x}`' for x in replace)+')' if replace else '*'
    derived=',\n'.join(f'  {expr} AS `{name}`' for name,expr in expressions.items())
    sql=f'''CREATE OR REPLACE TABLE `{A}` AS
SELECT {base},
{derived}
FROM `{A}`'''
    (SQL_DIR/f'stage3_08_spss_{block_name}.sql').write_text(sql,encoding='utf-8')
    job=client.query(sql,location=LOCATION)
    job.result()
    SPSS_MATERIALIZED.update(expressions)
    SPSS_BLOCK_AUDIT.append({
        'block':block_name,'required_columns':len(required),'output_columns':len(expressions),
        'sql_file':f'stage3_08_spss_{block_name}.sql',
        'sql_sha256':hashlib.sha256(sql.encode('utf-8')).hexdigest(),
        'bytes_processed':job.total_bytes_processed,'slot_millis':job.slot_millis,
        'run_utc':RUN_UTC})
    print(block_name,':',len(expressions),'columnas')

def sql_any1(names):
    return '('+' OR '.join(f'`{x}`=1' for x in names)+')'


## 4. Dimensiones y variables de contexto


In [5]:
# 1-8 y 10-12. Factores de desagregación y contexto.
factor_required={
    'C4P129','C3P128','C3P105','C3P126','C3P127','C3P123','C3P218','C3P219',
    'C3P220','C3P122','CCDD','CCPP','SEXO',
    *[f'C4P130_{i}' for i in range(1,7)],
    *[f'C3P115_{i}' for i in range(1,5)],
    *[f'C3P216A_{i}' for i in range(1,7)],
    *[f'C3P216A_{i}C' for i in range(1,7)],
    *[f'C3P216C_{i}' for i in range(1,6)],
    *[f'C3P216C_{i}C' for i in range(1,6)]}

parent_missing=' OR '.join(f'C3P115_{i} IS NULL' for i in range(1,5))
biparental=' OR '.join([
    '(C3P115_1=1 AND C3P115_2=1)','(C3P115_1=1 AND C3P115_4=1)',
    '(C3P115_2=1 AND C3P115_3=1)','(C3P115_3=1 AND C3P115_4=1)'])
monoparental=' OR '.join([
    f'(C3P115_{i}=1 AND '+ ' AND '.join(f'C3P115_{j}!=1' for j in range(1,5) if j!=i)+')'
    for i in range(1,5)])
personal_hit=' OR '.join(f'(C3P216A_{i}=1 AND C3P216A_{i}C=1)' for i in range(1,7))
induced_hit=' OR '.join(f'(C3P216C_{i}=1 AND C3P216C_{i}C=1)' for i in range(1,6))

factors={
 'etnicidad1':'''CASE
   WHEN C4P129 IN (1,2) THEN 1 WHEN C4P129 IN (3,4) THEN 3
   WHEN C4P129=5 THEN 5 WHEN C4P129 IN (6,7,8) THEN 6
   WHEN C4P129=9 THEN 9 END''',
 'idiomaHogar':'''CASE WHEN C3P128 IS NULL THEN -1 WHEN C3P128=1 THEN 1
   WHEN C3P128 IN (2,3) THEN 3 WHEN C3P128=4 THEN 4 END''',
 'DISCAPACIDAD':'''CASE
   WHEN C4P130_1=1 OR C4P130_2=1 OR C4P130_3=1 OR C4P130_4=1 OR C4P130_5=1 OR C4P130_6=1 THEN 1
   WHEN C4P130_1=2 AND C4P130_2=2 AND C4P130_3=2 AND C4P130_4=2 AND C4P130_5=2 AND C4P130_6=2 THEN 0 END''',
 'tipo_hogar1':f'''CASE
   WHEN C3P105=2 THEN 3
   WHEN C3P105!=2 AND ({biparental}) THEN 1
   WHEN C3P105!=2 AND ({monoparental}) THEN 2
   WHEN {parent_missing} THEN -1
   ELSE 3 END''',
 'DEPARTAMENTO2':'''CASE
   WHEN LPAD(CAST(CCDD AS STRING),2,'0')='15'
    AND LPAD(CAST(CCPP AS STRING),2,'0') IN ('02','03','04','05','06','07','08','09','10') THEN '64'
   ELSE LPAD(CAST(CCDD AS STRING),2,'0') END''',
 'OPINION_TOMADA':'''CASE WHEN C3P126=2 THEN 0
   WHEN C3P126=1 AND C3P127=2 THEN 1
   WHEN C3P126=1 AND C3P127=1 THEN 0 ELSE -1 END''',
 'discusion_hogar':'CASE WHEN C3P123=1 THEN 1 WHEN C3P123=2 THEN 0 END',
 'Desemp_DesaproboCurso':'CASE WHEN C3P218=1 THEN 1 WHEN C3P218=2 THEN 0 END',
 'Desemp_RepitioGrado':'CASE WHEN C3P219=1 THEN 1 WHEN C3P219=2 THEN 0 END',
 'Desemp_ExpulsionColegio':'CASE WHEN C3P220=1 THEN 1 WHEN C3P220=2 THEN 0 END',
 'conducta_riesgo_personal':f'CASE WHEN SEXO IN (1,2) AND ({personal_hit}) THEN 1 ELSE 0 END',
 'conducta_riesgo_inducido':f'CASE WHEN SEXO IN (1,2) AND ({induced_hit}) THEN 1 ELSE 0 END',
 'no_ir_colegio':'CASE WHEN C3P122=1 THEN 1 ELSE 0 END'}
apply_spss_block('31_factores',factor_required,factors)
apply_spss_block('31_riesgo_total',{'conducta_riesgo_personal','conducta_riesgo_inducido'},
                 {'riesgo_total':'CASE WHEN conducta_riesgo_personal=1 OR conducta_riesgo_inducido=1 THEN 1 ELSE 0 END'})


31_factores : 13 columnas
31_riesgo_total : 1 columnas


## 5. Percepciones, derechos y castigo físico


In [6]:
# Normas sobre castigo físico y reconocimiento de derechos.
attitude_required={f'C3P301_{i}' for i in [1,2,3,4,5,6]}
attitudes={
 'justifica_castigo_docente':'CASE WHEN C3P301_4=1 THEN 1 WHEN C3P301_4=2 THEN 0 END',
 'justifica_castigo_parental':'CASE WHEN C3P301_5=1 THEN 1 WHEN C3P301_5=2 THEN 0 END',
 'reconoce_derecho_opinar':'CASE WHEN C3P301_2=1 THEN 1 WHEN C3P301_2=2 THEN 0 END',
 'reconoce_derecho_denunciar':'CASE WHEN C3P301_6=1 THEN 1 WHEN C3P301_6=2 THEN 0 END',
 'rechaza_dejar_estudiar':'CASE WHEN C3P301_3=1 THEN 0 WHEN C3P301_3=2 THEN 1 END',
 'rechaza_trabajo_infantil_necesidad':'CASE WHEN C3P301_1=1 THEN 0 WHEN C3P301_1=2 THEN 1 END'}
apply_spss_block('31_actitudes_componentes',attitude_required,attitudes)

rights=['reconoce_derecho_opinar','reconoce_derecho_denunciar','rechaza_dejar_estudiar','rechaza_trabajo_infantil_necesidad']
rights_sum='+'.join(f'COALESCE({x},0)' for x in rights)
aggregate_attitudes={
 'justifica_al_menos_una':'''CASE
   WHEN justifica_castigo_docente IS NULL AND justifica_castigo_parental IS NULL THEN NULL
   ELSE GREATEST(COALESCE(justifica_castigo_docente,0),COALESCE(justifica_castigo_parental,0)) END''',
 'n_formas_justificadas':'''CASE
   WHEN justifica_castigo_docente IS NULL AND justifica_castigo_parental IS NULL THEN NULL
   ELSE COALESCE(justifica_castigo_docente,0)+COALESCE(justifica_castigo_parental,0) END''',
 'indice_derechos':f'''CASE WHEN {' AND '.join(f'{x} IS NULL' for x in rights)} THEN NULL ELSE {rights_sum} END''',
 'reconoce_todos_derechos_clave':f'''CASE WHEN {' OR '.join(f'{x} IS NULL' for x in rights)} THEN NULL
   WHEN {' AND '.join(f'{x}=1' for x in rights)} THEN 1 ELSE 0 END'''}
apply_spss_block('31_actitudes_agregados',set(attitudes),aggregate_attitudes)
apply_spss_block('31_reconoce_tres_mas',{'indice_derechos'},
                 {'reconoce_3omas_derechos':'CASE WHEN indice_derechos IS NULL THEN NULL WHEN indice_derechos>=3 THEN 1 ELSE 0 END'})


31_actitudes_componentes : 6 columnas
31_actitudes_agregados : 4 columnas
31_reconoce_tres_mas : 1 columnas


## 6. Roles de género y tareas del hogar


In [7]:
# Roles de género y división del trabajo del hogar.
task_required={f'C3P302_{i}' for i in range(1,11)}
task_expr={}
for i in range(1,8):
    source=f'C3P302_{i}'
    task_expr[f'tarea{i}_fem']=f'CASE WHEN {source} IN (2,4,6) THEN 1 WHEN {source} IN (1,3,5,7) THEN 0 END'
    task_expr[f'tarea{i}_masc']=f'CASE WHEN {source} IN (3,5,7) THEN 1 WHEN {source} IN (1,2,4,6) THEN 0 END'
    task_expr[f'tarea{i}_nna']=f'CASE WHEN {source}=1 THEN 1 WHEN {source} IN (2,3,4,5,6,7) THEN 0 END'
for i in range(8,11):
    source=f'C3P302_{i}'
    task_expr[f'tarea{i}_fem']=f'CASE WHEN {source} IN (2,4,6) THEN 1 WHEN {source} IN (3,5,7,8) THEN 0 END'
    task_expr[f'tarea{i}_masc']=f'CASE WHEN {source} IN (3,5,7) THEN 1 WHEN {source} IN (2,4,6,8) THEN 0 END'
    task_expr[f'tarea{i}_nadie']=f'CASE WHEN {source}=8 THEN 1 WHEN {source} IN (2,3,4,5,6,7) THEN 0 END'
apply_spss_block('31_tareas_componentes',task_required,task_expr)

fem=[f'tarea{i}_fem' for i in range(1,11)]
masc=[f'tarea{i}_masc' for i in range(1,11)]
nna=[f'tarea{i}_nna' for i in range(1,8)]
nadie=[f'tarea{i}_nadie' for i in range(8,11)]
valid_17='+'.join(f'CASE WHEN C3P302_{i} IN (1,2,3,4,5,6,7) THEN 1 ELSE 0 END' for i in range(1,8))
valid_810='+'.join(f'CASE WHEN C3P302_{i} IN (2,3,4,5,6,7,8) THEN 1 ELSE 0 END' for i in range(8,11))
def coalesce_sum(names): return '+'.join(f'COALESCE({x},0)' for x in names)
task_aggregates={
 'n_tareas_femeninas':coalesce_sum(fem),'n_tareas_masculinas':coalesce_sum(masc),
 'n_tareas_nna':coalesce_sum(nna),'n_tareas_nadie':coalesce_sum(nadie),
 'n_tareas_validas_1_7':valid_17,'n_tareas_validas_8_10':valid_810,
 'n_tareas_validas_p302':f'({valid_17})+({valid_810})'}
apply_spss_block('31_tareas_conteos',task_required|set(task_expr),task_aggregates)
apply_spss_block('31_tareas_indicador',{'n_tareas_femeninas','n_tareas_masculinas','n_tareas_validas_p302'},
 {'prop_tareas_femeninas':'SAFE_DIVIDE(n_tareas_femeninas,n_tareas_validas_p302)',
  'prop_tareas_masculinas':'SAFE_DIVIDE(n_tareas_masculinas,n_tareas_validas_p302)',
  'predominio_femenino_tareas':'''CASE WHEN n_tareas_validas_p302=0 THEN NULL
    WHEN n_tareas_femeninas>n_tareas_masculinas THEN 1 ELSE 0 END'''})


31_tareas_componentes : 30 columnas
31_tareas_conteos : 7 columnas
31_tareas_indicador : 3 columnas


## 7. Mitos sobre violencia sexual


In [8]:
# Mitos sobre violencia sexual.
myth_map={'mito_locas':'C3P303_1','mito_pobreza':'C3P303_3',
          'mito_fuera_casa':'C3P303_4','mito_sitios_oscuros':'C3P303_5'}
myths={name:f'CASE WHEN {source}=1 THEN 1 WHEN {source}=2 THEN 0 END' for name,source in myth_map.items()}
apply_spss_block('31_mitos_componentes',set(myth_map.values()),myths)
myth_names=list(myth_map)
myth_sum='+'.join(f'COALESCE({x},0)' for x in myth_names)
myth_all_missing=' AND '.join(f'{x} IS NULL' for x in myth_names)
apply_spss_block('31_mitos_agregados',set(myth_names),{
 'n_mitos':f'CASE WHEN {myth_all_missing} THEN NULL ELSE {myth_sum} END',
 'cree_al_menos_un_mito':f'CASE WHEN {myth_all_missing} THEN NULL WHEN {" OR ".join(f"{x}=1" for x in myth_names)} THEN 1 ELSE 0 END'})

print('PASS: SPSS 3.1 reconstruido exclusivamente desde variables fuente.')


31_mitos_componentes : 4 columnas
31_mitos_agregados : 2 columnas
PASS: SPSS 3.1 reconstruido exclusivamente desde variables fuente.


## 8. Funciones auxiliares para los módulos de violencia


In [9]:
# ============================================================
# SPSS 3.2-3.6: indicadores reconstruidos desde .sps
# Fuentes: 03, 05, 08, 09, 10, 11, 12 y 13 de CodigoSpss.
# Es reejecutable: conserva columnas ajenas y reemplaza solo sus salidas.
# ============================================================

A=globals().get('A',f'{PROJECT_ID}.enares2024_crs04_analytical.analytical_crs04_adolescents')

def apply_spss_block(block_name, required, expressions):
    """Materializa expresiones SPSS sin borrar columnas ajenas al bloque."""
    schema={f.name for f in client.get_table(A).schema}
    missing=sorted(set(required)-schema)
    if missing:
        raise RuntimeError(f'{block_name}: faltan fuentes SPSS: {missing}')
    aliases=list(expressions)
    replace=[x for x in aliases if x in schema]
    base='* EXCEPT('+','.join(f'`{x}`' for x in replace)+')' if replace else '*'
    derived=',\n'.join(f'  {expr} AS `{name}`' for name,expr in expressions.items())
    sql=f'''CREATE OR REPLACE TABLE `{A}` AS
SELECT {base},
{derived}
FROM `{A}`'''
    (SQL_DIR/f'stage3_08b_{block_name}.sql').write_text(sql,encoding='utf-8')
    job=client.query(sql,location=LOCATION)
    job.result()
    SPSS_MATERIALIZED.update(expressions)
    SPSS_BLOCK_AUDIT.append({
        'block':block_name,'required_columns':len(required),'output_columns':len(expressions),
        'sql_file':f'stage3_08b_{block_name}.sql',
        'sql_sha256':hashlib.sha256(sql.encode('utf-8')).hexdigest(),
        'bytes_processed':job.total_bytes_processed,'slot_millis':job.slot_millis,
        'run_utc':RUN_UTC})
    print(block_name,':',len(expressions),'columnas')

def any1(names):
    return '('+' OR '.join(f'`{x}`=1' for x in names)+')'

def all_present(names):
    return ' AND '.join(f'`{x}` IS NOT NULL' for x in names)


## 9. Violencia en el hogar


In [10]:
# ------------------------------------------------------------
# 3.2 Hogar: formas específicas e ICVAC
# ------------------------------------------------------------
detail_h={}
req_h={'SEXO','C3P203','C3P207'}
for stem,n,gateway in [('C3P201',11,'C3P203'),('C3P205',7,'C3P207')]:
    for i in range(1,n+1):
        item=f'{stem}_{i}'; A1=f'{stem}A_{i}'; E=f'{stem}E_{i}'
        C=f'{stem}C_{i}'; D=f'{stem}D_{i}'; F=f'{stem}F_{i}'
        req_h.update([item,A1,E,C,D,F])
        valid=f'''({A1} IN (1,2,3,4,19) OR {E} IN (1,2,3,4,19)
          OR ({A1} IS NOT NULL AND {A1} NOT IN (1,2,3,4,19) AND {C}=1 AND {D}=1)
          OR ({E} IS NOT NULL AND {E} NOT IN (1,2,3,4,19) AND {F}=1))'''
        detail_h[f'{stem}_{i}_1']=f'CASE WHEN SEXO IN (1,2) AND {item}=1 AND {gateway}=1 AND {valid} THEN 1 ELSE 0 END'
apply_spss_block('32_hogar_formas',req_h,detail_h)

# Prevalencias principales, negligencia y combinaciones del hogar. Todas se
# inicializan en 0 después de evaluar el IF, como los RECODE (SYSMIS=0) SPSS.
main_h={
 'VP_HOGAR':f'CASE WHEN {any1([f"C3P201_{i}_1" for i in range(1,12)])} THEN 1 ELSE 0 END',
 'VF_HOGAR':f'CASE WHEN {any1([f"C3P205_{i}_1" for i in range(1,8)])} THEN 1 ELSE 0 END',
 'VF_HOGAR_01':'CASE WHEN C3P121=1 THEN 1 ELSE 0 END',
 'VF_HOGAR_03':f'''CASE WHEN
   ({' OR '.join(f'(C3P216A_{i}=1 AND C3P216A_{i}C=1)' for i in range(1,7))}) OR
   ({' OR '.join(f'(C3P216C_{i}=1 AND C3P216C_{i}C=1)' for i in range(1,6))})
   THEN 1 ELSE 0 END'''}
req_main_h=set(detail_h)|{'C3P121'}
for i in range(1,7): req_main_h.update([f'C3P216A_{i}',f'C3P216A_{i}C'])
for i in range(1,6): req_main_h.update([f'C3P216C_{i}',f'C3P216C_{i}C'])
apply_spss_block('32_hogar_principales',req_main_h,main_h)
apply_spss_block('32_hogar_combinaciones_base',set(main_h),{
 'VN_HOGAR1':'CASE WHEN VF_HOGAR_01=1 OR VF_HOGAR_03=1 THEN 1 ELSE 0 END',
 'INDICADOR_8_3_6':'CASE WHEN VP_HOGAR=1 OR VF_HOGAR=1 OR VF_HOGAR_01=1 OR VF_HOGAR_03=1 THEN 1 ELSE 0 END',
 'VP_o_VF_HOGAR':'CASE WHEN VP_HOGAR=1 OR VF_HOGAR=1 THEN 1 ELSE 0 END',
 'VP_VF_HOGAR':'CASE WHEN VP_HOGAR=1 AND VF_HOGAR=1 THEN 1 ELSE 0 END'})
apply_spss_block('32_hogar_aliases_publicados',{'VP_VF_HOGAR'},
 {'Solap_VP_VF_H__Coexistencia':'VP_VF_HOGAR'})

icvac_h={
 'VP_ICVAC_401_ATERRORIZAR':f'CASE WHEN {any1(["C3P201_6_1","C3P201_7_1","C3P201_9_1"])} THEN 1 ELSE 0 END',
 'VP_ICVAC_402_HOSTIGAR_HUMILLAR':f'CASE WHEN {any1([f"C3P201_{i}_1" for i in range(1,6)])} THEN 1 ELSE 0 END',
 'VP_ICVAC_203_AISLAMIENTO':f'CASE WHEN {any1(["C3P201_8_1","C3P201_10_1"])} THEN 1 ELSE 0 END',
 'VP_ICVAC_409_OTROS':'CASE WHEN C3P201_11_1=1 THEN 1 ELSE 0 END',
 'VF_ICVAC_201_AGRESION_GRAVE':f'CASE WHEN {any1(["C3P205_5_1","C3P205_6_1"])} THEN 1 ELSE 0 END',
 'VF_ICVAC_202_AGRESION_LEVE':f'CASE WHEN {any1([f"C3P205_{i}_1" for i in range(1,5)])} THEN 1 ELSE 0 END',
 'VF_ICVAC_209_OTROS':'CASE WHEN C3P205_7_1=1 THEN 1 ELSE 0 END'}
apply_spss_block('32_hogar_icvac',detail_h,icvac_h)

# Agresores hogar: ocho grupos publicados por VP y VF.
def hogar_agresor_expr(stem,n,gateway,kind):
    groups={1:[1],2:[2],3:[3],4:[4],5:[5,6],6:[9,10],7:[18,19,20],8:[7,8,11,12,13,14,15,16,17]}
    out={}
    for g,codes in groups.items():
        hits=[]
        for i in range(1,n+1):
            item=f'{stem}_{i}'; a=f'{stem}A_{i}'; e=f'{stem}E_{i}'; c=f'{stem}C_{i}'; d=f'{stem}D_{i}'; f=f'{stem}F_{i}'
            code_list=','.join(map(str,codes))
            if g<=4:
                who=f'({a} IN ({code_list}) OR {e} IN ({code_list}))'
            elif g==7:
                who=f'(({a} IN (18,20) AND {c}=1 AND {d}=1) OR {a}=19 OR ({e} IN (18,19,20) AND {f}=1))'
            else:
                who=f'(({a} IN ({code_list}) AND {c}=1 AND {d}=1) OR ({e} IN ({code_list}) AND {f}=1))'
            hits.append(f'({item}=1 AND {gateway}=1 AND {who})')
        out[f'AG_{kind}_H_{g:02d}']='CASE WHEN '+' OR '.join(hits)+' THEN 1 ELSE 0 END'
    return out

ag_h={**hogar_agresor_expr('C3P201',11,'C3P203','VP'),**hogar_agresor_expr('C3P205',7,'C3P207','VF')}
apply_spss_block('321_hogar_agresores',req_h,ag_h)


32_hogar_formas : 18 columnas
32_hogar_principales : 4 columnas
32_hogar_combinaciones_base : 4 columnas
32_hogar_aliases_publicados : 1 columnas
32_hogar_icvac : 7 columnas
321_hogar_agresores : 16 columnas


## 10. Violencia en la escuela


In [11]:
# ------------------------------------------------------------
# 3.3 Escuela: formas, prevalencias, ICVAC y agresores
# ------------------------------------------------------------
detail_e={}; req_e={'C3P225','C3P229'}
for stem,n,gateway in [('C3P223',14,'C3P225'),('C3P227',10,'C3P229')]:
    for i in range(1,n+1):
        cols=[f'{stem}_{i}',f'{stem}A_{i}',f'{stem}C_{i}',f'{stem}E_{i}']; req_e.update(cols)
        detail_e[f'{stem}_{i}_1']=f'CASE WHEN {cols[0]}=1 AND {gateway}=1 AND ({cols[1]}=1 OR {cols[2]}=1 OR {cols[3]}=1) THEN 1 ELSE 0 END'
apply_spss_block('33_escuela_formas',req_e,detail_e)

school_main={
 'VP_ESCUELA':f'CASE WHEN {any1([f"C3P223_{i}_1" for i in range(1,15)])} THEN 1 ELSE 0 END',
 'VF_ESCUELA':f'CASE WHEN {any1([f"C3P227_{i}_1" for i in range(1,11)])} THEN 1 ELSE 0 END',
 'VPE_ICVAC_401_ATERRORIZAR':f'CASE WHEN {any1(["C3P223_12_1","C3P223_13_1"])} THEN 1 ELSE 0 END',
 'VPE_ICVAC_402_HOSTIGAR_HUMILLAR':f'CASE WHEN {any1([f"C3P223_{i}_1" for i in range(1,11)])} THEN 1 ELSE 0 END',
 'VPE_ICVAC_203_AISLAMIENTO':'CASE WHEN C3P223_11_1=1 THEN 1 ELSE 0 END',
 'VPE_ICVAC_409_OTROS':'CASE WHEN C3P223_14_1=1 THEN 1 ELSE 0 END',
 'VFE_ICVAC_201_AGRESION_GRAVE':f'CASE WHEN {any1([f"C3P227_{i}_1" for i in [5,7,8,9]])} THEN 1 ELSE 0 END',
 'VFE_ICVAC_202_AGRESION_LEVE':f'CASE WHEN {any1([f"C3P227_{i}_1" for i in [1,2,3,4,6]])} THEN 1 ELSE 0 END',
 'VFE_ICVAC_209_OTROS':'CASE WHEN C3P227_10_1=1 THEN 1 ELSE 0 END'}
apply_spss_block('33_escuela_principales',detail_e,school_main)

def escuela_agresor_expr(stem,n,gateway,kind):
    specs={1:('A',None),2:('A',1),3:('A',2),4:('A',3),5:('C',None),6:('C',1),7:('C',2),8:('C',3),9:('E',None)}
    out={}
    for g,(who,age) in specs.items():
        hits=[]
        for i in range(1,n+1):
            person=f'{stem}{who}_{i}'; agevar=f'{stem}{"B" if who=="A" else "D"}_{i}' if who in {'A','C'} else None
            cond=f'{stem}_{i}=1 AND {gateway}=1 AND {person}=1'
            if age is not None: cond+=f' AND {agevar}={age}'
            hits.append('('+cond+')')
        out[f'AG_{kind}_E_{g:02d}']='CASE WHEN '+' OR '.join(hits)+' THEN 1 ELSE 0 END'
    return out

req_ag_e=set(req_e)
for stem,n in [('C3P223',14),('C3P227',10)]:
    for i in range(1,n+1): req_ag_e.update([f'{stem}B_{i}',f'{stem}D_{i}'])
ag_e={**escuela_agresor_expr('C3P223',14,'C3P225','VP'),**escuela_agresor_expr('C3P227',10,'C3P229','VF')}
apply_spss_block('331_escuela_agresores',req_ag_e,ag_e)


33_escuela_formas : 24 columnas
33_escuela_principales : 9 columnas
331_escuela_agresores : 18 columnas


## 11. Violencia sexual y agresores


In [12]:
# Violencia sexual general, hogar y escuela: vida y últimos 12 meses.
req_vs={'SEXO'}; detail_vs={}
for i in range(1,17):
    req_vs.update([f'C4P248_{i}',f'C4P248C_{i}'])
    detail_vs[f'P248_{i:02d}_12M']=f'CASE WHEN C4P248_{i}=1 AND C4P248C_{i}=1 THEN 1 ELSE 0 END'
    for j in range(1,29): req_vs.add(f'C4P248A_{j}_{i}')
    detail_vs[f'C4P248_{i}_1']=f'CASE WHEN C4P248_{i}=1 AND C4P248C_{i}=1 AND (C4P248A_27_{i}=1 OR C4P248A_28_{i}=1) THEN 1 ELSE 0 END'
    detail_vs[f'C4P248_{i}_1_1']=f'CASE WHEN C4P248_{i}=1 AND (C4P248A_27_{i}=1 OR C4P248A_28_{i}=1) THEN 1 ELSE 0 END'
    detail_vs[f'C4P248_{i}_3']=f'CASE WHEN C4P248_{i}=1 AND C4P248C_{i}=1 AND ({" OR ".join(f"C4P248A_{j}_{i}=1" for j in range(1,18))}) THEN 1 ELSE 0 END'
    detail_vs[f'C4P248_{i}_3_1']=f'CASE WHEN C4P248_{i}=1 AND ({" OR ".join(f"C4P248A_{j}_{i}=1" for j in range(1,18))}) THEN 1 ELSE 0 END'
apply_spss_block('34_vs_formas_contexto',req_vs,detail_vs)

vs_main={
 'VS_12M':f'CASE WHEN {any1([f"P248_{i:02d}_12M" for i in range(1,17)])} THEN 1 ELSE 0 END',
 'VS_VIDA':f'CASE WHEN {any1([f"C4P248_{i}" for i in range(1,17)])} THEN 1 ELSE 0 END',
 'VS_E':f'CASE WHEN {any1([f"C4P248_{i}_1" for i in range(1,17)])} THEN 1 ELSE 0 END',
 'VS_E_1':f'CASE WHEN {any1([f"C4P248_{i}_1_1" for i in range(1,17)])} THEN 1 ELSE 0 END',
 'VS_H':f'CASE WHEN {any1([f"C4P248_{i}_3" for i in range(1,17)])} THEN 1 ELSE 0 END',
 'VS_H_1':f'CASE WHEN {any1([f"C4P248_{i}_3_1" for i in range(1,17)])} THEN 1 ELSE 0 END'}
apply_spss_block('34_vs_principales',detail_vs,vs_main)

def grouped_binary(source_prefix,suffix,indices):
    return f'CASE WHEN {any1([f"{source_prefix}{i}{suffix}" for i in indices])} THEN 1 ELSE 0 END'

vs_groups={}
groups={'301':[11],'302':[4,5,6],'303':[1,2,3,7,8,9,10,13,14,15,16],'309':[12]}
cp={'CP01':[1,2],'CP02':[3,7,9],'CP03':[4,5,6,8],'CP04':[10],'CP05':[11],'CP06':[13],'CP07':[14],'CP08':[15,16],'CP09':[12]}
for code,idx in groups.items():
    vs_groups[f'VS_ICVAC_{code}']=grouped_binary('P248_','_12M',idx) if False else f'CASE WHEN {any1([f"P248_{i:02d}_12M" for i in idx])} THEN 1 ELSE 0 END'
    vs_groups[f'VS_ICVAC_{code}_VIDA']=f'CASE WHEN {any1([f"C4P248_{i}" for i in idx])} THEN 1 ELSE 0 END'
    vs_groups[f'VSE_{code}']=f'CASE WHEN {any1([f"C4P248_{i}_1" for i in idx])} THEN 1 ELSE 0 END'
    vs_groups[f'VSE1_{code}']=f'CASE WHEN {any1([f"C4P248_{i}_1_1" for i in idx])} THEN 1 ELSE 0 END'
    vs_groups[f'VSH_{code}']=f'CASE WHEN {any1([f"C4P248_{i}_3" for i in idx])} THEN 1 ELSE 0 END'
    vs_groups[f'VSH1_{code}']=f'CASE WHEN {any1([f"C4P248_{i}_3_1" for i in idx])} THEN 1 ELSE 0 END'
for code,idx in cp.items():
    vs_groups[f'VS_{code}']=f'CASE WHEN {any1([f"P248_{i:02d}_12M" for i in idx])} THEN 1 ELSE 0 END'
    vs_groups[f'VS1_{code}']=f'CASE WHEN {any1([f"C4P248_{i}" for i in idx])} THEN 1 ELSE 0 END'
    vs_groups[f'VSE_{code}']=f'CASE WHEN {any1([f"C4P248_{i}_1" for i in idx])} THEN 1 ELSE 0 END'
    vs_groups[f'VSE1_{code}']=f'CASE WHEN {any1([f"C4P248_{i}_1_1" for i in idx])} THEN 1 ELSE 0 END'
    vs_groups[f'VSH_{code}']=f'CASE WHEN {any1([f"C4P248_{i}_3" for i in idx])} THEN 1 ELSE 0 END'
    vs_groups[f'VSH1_{code}']=f'CASE WHEN {any1([f"C4P248_{i}_3_1" for i in idx])} THEN 1 ELSE 0 END'
apply_spss_block('34_vs_icvac_cp',detail_vs,vs_groups)
apply_spss_block('34_vs_contacto',{'VS_ICVAC_301','VS_ICVAC_302','VS_ICVAC_301_VIDA','VS_ICVAC_302_VIDA'},
 {'VS_ICVAC_CONTACTO':'CASE WHEN VS_ICVAC_301=1 OR VS_ICVAC_302=1 THEN 1 ELSE 0 END',
  'VS_ICVAC_CONTACTO_VIDA':'CASE WHEN VS_ICVAC_301_VIDA=1 OR VS_ICVAC_302_VIDA=1 THEN 1 ELSE 0 END'})

# Agresores de violencia sexual (nueve grupos publicados).
ag_codes={1:list(range(1,18)),2:[1,3,5,7,9,11,13,15],3:[2,4,6,8,10,12,14,16],4:[17],5:[27,28],6:[25],7:[26],8:[22,23,24],9:[18,19,20,21]}
ag_vs={}
for g,codes in ag_codes.items():
    recent=[]; life=[]
    for i in range(1,17):
        who=' OR '.join(f'C4P248A_{j}_{i}=1' for j in codes)
        recent.append(f'(C4P248_{i}=1 AND C4P248C_{i}=1 AND ({who}))')
        life.append(f'(C4P248_{i}=1 AND ({who}))')
    ag_vs[f'AG_VS12_{g:02d}']='CASE WHEN '+' OR '.join(recent)+' THEN 1 ELSE 0 END'
    ag_vs[f'AG_VSVIDA_{g:02d}']='CASE WHEN '+' OR '.join(life)+' THEN 1 ELSE 0 END'
apply_spss_block('34_vs_agresores',req_vs,ag_vs)

# Indicadores 8.2.7, 8.2.13 y 8.2.8. SPSS los estima en el dominio SEXO=1;
# aquí se materializa el numerador 0/1 y el dominio se declara en 09.
req_vs_mujer={'SEXO','C4P248_O_12'}
for i in range(1,17):
    req_vs_mujer.update([f'C4P248_{i}',f'C4P248A_26_{i}',f'C4P248C_{i}',f'C4P248B_{i}'])
no_pareja_12m=[f'(C4P248_{i}=1 AND C4P248A_26_{i}!=1 AND C4P248C_{i}=1)' for i in range(1,17)]
acoso_idx=[1,2,3,4,6,7,9,13,14]
acoso_12m=[f'(C4P248_{i}=1 AND C4P248A_26_{i}!=1 AND C4P248C_{i}=1)' for i in acoso_idx]
acoso_12m.append('(UPPER(TRIM(CAST(C4P248_O_12 AS STRING)))="ACOSO SEXUAL" AND C4P248A_26_12!=1 AND C4P248C_12=1)')
antes_12=[f'(C4P248_{i}=1 AND C4P248B_{i}<12)' for i in range(1,17)]
ind_vs_mujer={
 'INDICADOR_8_2_7':'CASE WHEN SEXO=1 AND ('+' OR '.join(no_pareja_12m)+') THEN 1 ELSE 0 END',
 'INDICADOR_8_2_13':'CASE WHEN SEXO=1 AND ('+' OR '.join(acoso_12m)+') THEN 1 ELSE 0 END',
 'INDICADOR_8_2_8':'CASE WHEN SEXO=1 AND ('+' OR '.join(antes_12)+') THEN 1 ELSE 0 END'}
apply_spss_block('34_vs_indicadores_mujeres',req_vs_mujer,ind_vs_mujer)

# Aliases exactos de los IDs SPSS con prefijo contextual.
aliases={}
for i in range(1,17):
    aliases[f'Formas_VS_12M__C4P248_{i}']=f'P248_{i:02d}_12M'
    aliases[f'Formas_VS_VIDA__C4P248_{i}']=f'CASE WHEN C4P248_{i}=1 THEN 1 ELSE 0 END'
    aliases[f'Formas_VS_E__C4P248_{i}']=f'C4P248_{i}_1'
    aliases[f'Formas_VS_E_1__C4P248_{i}']=f'C4P248_{i}_1_1'
    if i!=12:
        aliases[f'Formas_Agresor_VS_H__C4P248_{i}']=f'C4P248_{i}_3'
        aliases[f'Formas_Agresor_VS_H_1__C4P248_{i}']=f'C4P248_{i}_3_1'
for code in ['301','302','303','309']:
    aliases[f'Formas_VS_12M__ICVAC_{code}']=f'VS_ICVAC_{code}'
    aliases[f'Formas_VS_VIDA__ICVAC_{code}']=f'VS_ICVAC_{code}_VIDA'
    aliases[f'Formas_VS_E__ICVAC_{code}']=f'VSE_{code}'
    aliases[f'Formas_VS_E_1__ICVAC_{code}']=f'VSE1_{code}'
    if code!='309':
        aliases[f'Formas_Agresor_VS_H__ICVAC_{code}']=f'VSH_{code}'
        aliases[f'Formas_Agresor_VS_H_1__ICVAC_{code}']=f'VSH1_{code}'
for i in range(1,10):
    code=f'CP{i:02d}'
    aliases[f'Formas_VS_12M__VS_{code}']=f'VS_{code}'
    aliases[f'Formas_VS_VIDA__VS_{code}']=f'VS1_{code}'
    aliases[f'Formas_VS_E__VSE_{code}']=f'VSE_{code}'
    aliases[f'Formas_VS_E_1__VSE_{code}']=f'VSE1_{code}'
    if i<=8:
        aliases[f'Formas_Agresor_VS_H__VSH_{code}']=f'VSH_{code}'
        aliases[f'Formas_Agresor_VS_H_1__VSH_{code}']=f'VSH1_{code}'
for i in range(1,9):
    aliases[f'Agresor_VP_E__AG_VP_{i:02d}']=f'AG_VP_E_{i:02d}'
    aliases[f'Agresor_VF_E__AG_VF_{i:02d}']=f'AG_VF_E_{i:02d}'
    aliases[f'Agresor_VP_H__AG_VP_{i:02d}']=f'AG_VP_H_{i:02d}'
    aliases[f'Agresor_VF_H__AG_VF_{i:02d}']=f'AG_VF_H_{i:02d}'
# Los IDs sin prefijo solo aparecen para el noveno grupo escolar (otra persona).
aliases['AG_VP_09']='AG_VP_E_09'
aliases['AG_VF_09']='AG_VF_E_09'
for i in range(1,10):
    aliases[f'Agresor_VS_12M__AG_{i:02d}']=f'AG_VS12_{i:02d}'
    aliases[f'Agresor_VS_VIDA__AG_{i:02d}']=f'AG_VSVIDA_{i:02d}'
    # La salida consolidada de prevalencias conserva VIDA para los grupos
    # 01-07 y 12M para adultos del colegio/CAR (08-09).
    prev_source='AG_VSVIDA' if i<=7 else 'AG_VS12'
    aliases[f'Prev_Agresor_VS__AG_{i:02d}']=f'{prev_source}_{i:02d}'
# En las tablas del hogar, AG01/02/03 son familiar mujer, familiar hombre y
# otro ascendiente: equivalen a los grupos globales 02/03/04.
for published,group in [('AG01','02'),('AG02','03'),('AG03','04')]:
    aliases[f'Formas_Agresor_VS_H__VSH_{published}']=f'AG_VS12_{group}'
    aliases[f'Formas_Agresor_VS_H_1__VSH_{published}']=f'AG_VSVIDA_{group}'
aliases['VS_OtraPersona_12M']='AG_VS12_06'
aliases['VS_OtraPersona_VIDA']='AG_VSVIDA_06'
alias_required={expr for expr in aliases.values() if expr.replace('_','').isalnum()}
apply_spss_block('34_aliases_spss',alias_required,aliases)


34_vs_formas_contexto : 80 columnas
34_vs_principales : 6 columnas
34_vs_icvac_cp : 78 columnas
34_vs_contacto : 2 columnas
34_vs_agresores : 18 columnas
34_vs_indicadores_mujeres : 3 columnas
34_aliases_spss : 237 columnas


## 12. Lugares, horarios y acumulación de violencias


In [13]:
# Escuela ejercida, lugares y horarios.
req_ctx={'C3P233A',*[f'C3P233_{x}' for x in ['1','1A','2','2A','3']],*[f'C3P231_{i}' for i in range(1,7)]}
for i in range(1,6):
    for e in range(1,5): req_ctx.add(f'C3P231_{i}E{e}')
ctx={}
for x in ['1','1A','2','2A','3']:
    ctx[f'C3P233_{x}_1']=f'CASE WHEN C3P233_{x}=1 AND C3P233A=1 THEN 1 ELSE 0 END'
for name,i in [('LugarSalon_ViolEsc',1),('LugarPatio_ViolEsc',2),('LugarBano_ViolEsc',3),('LugarPasilloEscalera_ViolEsc',4),('LugarOtro_ViolEsc',5),('LugarFueraColegio_ViolEsc',6)]:
    ctx[name]=f'CASE WHEN C3P231_{i}=1 THEN 1 ELSE 0 END'
for name,e in [('HoraEntrada_ViolEsc',1),('HoraClase_ViolEsc',2),('HoraRecreo_ViolEsc',3),('HoraSalida_ViolEsc',4)]:
    ctx[name]=f'CASE WHEN {any1([f"C3P231_{i}E{e}" for i in range(1,6)])} THEN 1 ELSE 0 END'
apply_spss_block('336_escuela_contexto',req_ctx,ctx)
exercise={
 'VP_EJERCIDA':f'CASE WHEN {any1(["C3P233_2_1","C3P233_2A_1","C3P233_3_1"])} THEN 1 ELSE 0 END',
 'VF_EJERCIDA':f'CASE WHEN {any1(["C3P233_1_1","C3P233_1A_1"])} THEN 1 ELSE 0 END'}
apply_spss_block('336_escuela_ejercida',ctx,exercise)

# Combinaciones y acumulación 3.3/3.5.
combo={
 'VP_o_VF_E':'CASE WHEN VP_ESCUELA=1 OR VF_ESCUELA=1 THEN 1 ELSE 0 END',
 'VP_VF_E':'CASE WHEN VP_ESCUELA=1 AND VF_ESCUELA=1 THEN 1 ELSE 0 END',
 'VP_VF_VS_E':'CASE WHEN VP_ESCUELA=1 AND VF_ESCUELA=1 AND VS_E=1 THEN 1 ELSE 0 END',
 'VP_o_VF_EJERCIDA':'CASE WHEN VP_EJERCIDA=1 OR VF_EJERCIDA=1 THEN 1 ELSE 0 END',
 'INDICADOR_8_3_9':'CASE WHEN VP_ESCUELA=1 OR VF_ESCUELA=1 OR VS_E=1 THEN 1 ELSE 0 END'}
apply_spss_block('33_escuela_combinaciones',{'VP_ESCUELA','VF_ESCUELA','VS_E','VP_EJERCIDA','VF_EJERCIDA'},combo)
apply_spss_block('33_escuela_aliases_publicados',{'VP_o_VF_E','VP_VF_E'},
 {'VP_o_VF_ESCUELA':'VP_o_VF_E',
  'Solap_VP_VF_E__Coexistencia':'VP_VF_E'})
# Combinaciones con violencia sexual en el hogar. VS_H reproduce el VS_HOGAR
# de la sintaxis 08 (últimos 12 meses y agresor del ámbito del hogar).
apply_spss_block('32_hogar_tres_formas',{'VP_HOGAR','VF_HOGAR','VS_H'},
 {'VS_HOGAR':'VS_H',
  'VP_VF_VS_HOGAR':'CASE WHEN VP_HOGAR=1 AND VF_HOGAR=1 AND VS_H=1 THEN 1 ELSE 0 END',
  'VP_o_VF_o_VS_HOGAR':'CASE WHEN VP_HOGAR=1 OR VF_HOGAR=1 OR VS_H=1 THEN 1 ELSE 0 END'})
acc={
 'PV_hogar_escuela1':'CASE WHEN VP_o_VF_o_VS_HOGAR=1 AND INDICADOR_8_3_9=1 THEN 1 ELSE 0 END',
 'PV_hogar_escuela':'CASE WHEN VP_o_VF_HOGAR=1 AND VP_o_VF_E=1 THEN 1 ELSE 0 END',
 'PV_VP_hogar_escuela':'CASE WHEN VP_HOGAR=1 AND VP_ESCUELA=1 THEN 1 ELSE 0 END',
 'PV_VF_hogar_escuela':'CASE WHEN VF_HOGAR=1 AND VF_ESCUELA=1 THEN 1 ELSE 0 END',
 'PV_VP_hogar_VF_escuela':'CASE WHEN VP_HOGAR=1 AND VF_ESCUELA=1 THEN 1 ELSE 0 END',
 'PV_VF_hogar_VP_escuela':'CASE WHEN VF_HOGAR=1 AND VP_ESCUELA=1 THEN 1 ELSE 0 END',
 'PV_VP_VF_hogar_escuela':'CASE WHEN VP_HOGAR=1 AND VF_HOGAR=1 AND VP_ESCUELA=1 AND VF_ESCUELA=1 THEN 1 ELSE 0 END',
 'PV_VP_VF_hogar_escuela_VS':'CASE WHEN VP_HOGAR=1 AND VF_HOGAR=1 AND VP_ESCUELA=1 AND VF_ESCUELA=1 AND VS_12M=1 THEN 1 ELSE 0 END',
 'PV_indice_acum':'COALESCE(VP_HOGAR,0)+COALESCE(VF_HOGAR,0)+COALESCE(VP_ESCUELA,0)+COALESCE(VF_ESCUELA,0)',
 'PV_indice_acum_VS':'COALESCE(VP_HOGAR,0)+COALESCE(VF_HOGAR,0)+COALESCE(VP_ESCUELA,0)+COALESCE(VF_ESCUELA,0)+COALESCE(VS_12M,0)'}
apply_spss_block('35_acumulacion',{'VP_HOGAR','VF_HOGAR','VP_ESCUELA','VF_ESCUELA','VS_12M','VP_o_VF_HOGAR','VP_o_VF_E','VP_o_VF_o_VS_HOGAR','INDICADOR_8_3_9'},acc)


336_escuela_contexto : 15 columnas
336_escuela_ejercida : 2 columnas
33_escuela_combinaciones : 5 columnas
33_escuela_aliases_publicados : 2 columnas
32_hogar_tres_formas : 3 columnas
35_acumulacion : 10 columnas


## 13. Consecuencias físicas y atención de salud


In [14]:
# Consecuencias físicas: SPSS exige las seis respuestas válidas. La atención
# de salud solo tiene denominador entre quienes presentaron alguna consecuencia.
cons_sources={f'C3P243_{i}' for i in range(1,7)} | {f'C3P243_T{i}' for i in range(1,7)}
cons_valid=' AND '.join(f'C3P243_{i} IS NOT NULL' for i in range(1,7))
cons_any=any1([f'C3P243_{i}' for i in range(1,7)])
cons_count='+'.join(f'CASE WHEN C3P243_{i}=1 THEN 1 ELSE 0 END' for i in range(1,7))
cons_health=any1([f'C3P243_T{i}' for i in range(1,7)])
cons_expr={
 'CONS_ALGUNA':f'CASE WHEN NOT ({cons_valid}) THEN NULL WHEN {cons_any} THEN 1 ELSE 0 END',
 'CONS_NUM_CONSECUENCIAS':f'CASE WHEN NOT ({cons_valid}) THEN NULL ELSE {cons_count} END'}
apply_spss_block('354_consecuencias_base',cons_sources,cons_expr)
apply_spss_block('354_consecuencias_salud',cons_sources|{'CONS_ALGUNA'},
 {'CONS_ATENCION_SALUD':f'CASE WHEN CONS_ALGUNA!=1 OR CONS_ALGUNA IS NULL THEN NULL WHEN {cons_health} THEN 1 ELSE 0 END'})


354_consecuencias_base : 2 columnas
354_consecuencias_salud : 1 columnas


## 14. Búsqueda y recepción de ayuda


In [15]:
# ------------------------------------------------------------
# 3.6 Búsqueda de ayuda: hogar, escuela y violencia sexual
# ------------------------------------------------------------
help_expr={}; req_help=set()
def recode12(source): return f'CASE WHEN {source}=1 THEN 1 WHEN {source}=2 THEN 0 END'
for context,prefix,qsearch,qpeople,qreceived,qresponse,qinst in [
 ('hogar','ayuda_hogar','C3P209','C3P210','C3P211','C3P212','C3P214'),
 ('escuela','ayuda_escuela','C3P236','C3P237','C3P238','C3P239','C3P241')]:
    req_help.update([qsearch,qreceived,qinst])
    help_expr[f'busco_ayuda_{context}']=recode12(qsearch)
    help_expr[f'recibio_ayuda_{context}']=f'CASE WHEN {qreceived}=1 THEN 1 WHEN {qreceived}=2 THEN 0 END'
    help_expr[f'apoyo_institucional_{context}']=recode12(qinst)
    people={'madre':[1],'padre':[2],'madrastra':[3],'padrastro':[4],'hermana':[5],'hermano':[6],'abuela':[7],'abuelo':[8],'tia':[9],'tio':[10],'otro_pariente':[11],'familiar':list(range(1,12)),'car':[12,13,14],'escolar_adulto':[15,16,17,18],'pares_amigos':[19,20],'otro':[21]}
    for label,idx in people.items():
        cols=[f'{qpeople}_{i}' for i in idx]; req_help.update(cols)
        help_expr[f'{prefix}_{label}']=f'CASE WHEN {qsearch} IN (1,2) AND {any1(cols)} THEN 1 WHEN {qsearch} IN (1,2) THEN 0 END'
    responses=(['consuelo','consejo','hablo_familia','llamo_atencion','respuesta_agresion','otro_tipo'] if context=='hogar'
               else ['consuelo','aviso_docente','hablo_agresor','llamo_atencion','hablo_director','hablo_padres_agresor','consejo','otro_tipo'])
    for i,label in enumerate(responses,1):
        col=f'{qresponse}_{i}';req_help.add(col);help_expr[f'{prefix}_{label}']=f'CASE WHEN {col}=1 THEN 1 ELSE 0 END'
apply_spss_block('36_ayuda_componentes',req_help,help_expr)

help_derived={
 'recibio_ayuda_hogar_victimas':'CASE WHEN (VP_HOGAR=1 OR VF_HOGAR=1) AND C3P211=3 THEN NULL WHEN VP_HOGAR=1 OR VF_HOGAR=1 THEN CASE WHEN C3P211=1 THEN 1 ELSE 0 END END',
 'brecha_ayuda_hogar':'CASE WHEN busco_ayuda_hogar=1 AND recibio_ayuda_hogar=0 THEN 1 WHEN busco_ayuda_hogar=1 AND recibio_ayuda_hogar=1 THEN 0 END',
 'brecha_institucional_hogar':'CASE WHEN (VP_HOGAR=1 OR VF_HOGAR=1) AND apoyo_institucional_hogar=0 THEN 1 WHEN (VP_HOGAR=1 OR VF_HOGAR=1) AND apoyo_institucional_hogar=1 THEN 0 END',
 'recibio_ayuda_escuela_victimas':'CASE WHEN (VP_ESCUELA=1 OR VF_ESCUELA=1) AND C3P238=3 THEN NULL WHEN VP_ESCUELA=1 OR VF_ESCUELA=1 THEN CASE WHEN C3P238=1 THEN 1 ELSE 0 END END',
 'brecha_ayuda_escuela':'CASE WHEN busco_ayuda_escuela=1 AND recibio_ayuda_escuela=0 THEN 1 WHEN busco_ayuda_escuela=1 AND recibio_ayuda_escuela=1 THEN 0 END',
 'brecha_institucional_escuela':'CASE WHEN (VP_ESCUELA=1 OR VF_ESCUELA=1) AND apoyo_institucional_escuela=0 THEN 1 WHEN (VP_ESCUELA=1 OR VF_ESCUELA=1) AND apoyo_institucional_escuela=1 THEN 0 END'}
apply_spss_block('36_ayuda_derivados',set(help_expr)|{'VP_HOGAR','VF_HOGAR','VP_ESCUELA','VF_ESCUELA','C3P211','C3P238'},help_derived)

req_vs_help={'C4P252','C4P254','C4P257','C4P259','C3P246','C3P247'}
vs_help={'busco_ayuda_vs':'CASE WHEN C4P252=1 THEN 1 WHEN C4P252=2 THEN 0 END','recibio_ayuda_vs':'CASE WHEN C4P254=1 THEN 1 WHEN C4P254=2 THEN 0 END','apoyo_institucional_vs':recode12('C4P257'),'recibio_ayuda_institucional_vs':'CASE WHEN C4P259=1 THEN 1 WHEN C4P259=2 THEN 0 END','conoce_demuna':recode12('C3P246'),'uso_demuna':recode12('C3P247')}
people_vs={'madre':[1],'padre':[2],'madrastra':[3],'padrastro':[4],'hermana':[5],'hermano':[6],'abuela':[7],'abuelo':[8],'tia':[9],'tio':[10],'otro_pariente':[11],'familiar':list(range(1,12)),'car':[12,13,14],'escolar_adulto':[15,16,17,18],'pares_amigos':[19,20],'otro':[21]}
for label,idx in people_vs.items():
    cols=[f'C4P253_{i}' for i in idx];req_vs_help.update(cols);vs_help[f'ayuda_vs_{label}']=f'CASE WHEN C4P252 IN (1,2) AND {any1(cols)} THEN 1 WHEN C4P252 IN (1,2) THEN 0 END'
for i,label in enumerate(['consejo','hablo_madre_padre','reclamo_agresor','aviso_autoridades','refugio','especialista','otro_tipo'],1):
    col=f'C4P255_{i}';req_vs_help.add(col);vs_help[f'ayuda_vs_{label}']=f'CASE WHEN {col}=1 THEN 1 ELSE 0 END'
for i,label in enumerate(['hablo_familia','terapias','llamo_atencion','otro'],1):
    col=f'C4P260_{i}';req_vs_help.add(col);vs_help[f'ayuda_inst_vs_{label}']=f'CASE WHEN {col}=1 THEN 1 ELSE 0 END'
apply_spss_block('36_ayuda_vs_componentes',req_vs_help,vs_help)
vs_help2={
 'recibio_ayuda_vs_victimas':'CASE WHEN VS_12M=1 AND C4P254=3 THEN NULL WHEN VS_12M=1 THEN CASE WHEN C4P254=1 THEN 1 ELSE 0 END END',
 'brecha_ayuda_vs':'CASE WHEN busco_ayuda_vs=1 AND recibio_ayuda_vs=0 THEN 1 WHEN busco_ayuda_vs=1 AND recibio_ayuda_vs=1 THEN 0 END',
 'brecha_institucional_vs':'CASE WHEN VS_12M=1 AND apoyo_institucional_vs=0 THEN 1 WHEN VS_12M=1 AND apoyo_institucional_vs=1 THEN 0 END'}
apply_spss_block('36_ayuda_vs_derivados',set(vs_help)|{'VS_12M','C4P254'},vs_help2)

print('PASS: módulos SPSS 3.2-3.6 materializados en 08B.')


36_ayuda_componentes : 52 columnas
36_ayuda_derivados : 6 columnas
36_ayuda_vs_componentes : 33 columnas
36_ayuda_vs_derivados : 3 columnas
PASS: módulos SPSS 3.2-3.6 materializados en 08B.


## 15. Validación, linaje, métricas y registro de ejecución


In [16]:
schema = {field.name for field in client.get_table(A).schema}
missing = sorted(SPSS_MATERIALIZED - schema)
if missing:
    raise RuntimeError(f'Faltan columnas materializadas: {missing}')

checks = client.query(f'''SELECT
  COUNT(*) AS filas,
  COUNT(DISTINCT ID) AS conglomerados,
  COUNTIF(VP_HOGAR NOT IN (0,1) OR VF_HOGAR NOT IN (0,1)) AS errores_hogar,
  COUNTIF(OPINION_TOMADA NOT IN (-1,0,1)) AS errores_opinion,
  COUNTIF(conducta_riesgo_personal NOT IN (0,1)
          OR conducta_riesgo_inducido NOT IN (0,1)) AS errores_riesgo
FROM `{A}`''').result().to_dataframe()
display(checks)

row = checks.iloc[0]
valid = (
    row.filas == EXPECTED_ROWS
    and key_check.iloc[0].distinct_keys == EXPECTED_ROWS
    and row.conglomerados > 0
    and row.errores_hogar == 0
    and row.errores_opinion == 0
    and row.errores_riesgo == 0
)
if not valid:
    raise RuntimeError('Falló la validación final de los indicadores SPSS.')

manifest = pd.DataFrame({
    'column': sorted(SPSS_MATERIALIZED),
    'authority': 'SPSS syntax',
    'run_utc': RUN_UTC,
})
manifest.to_csv(LOG_DIR / 'stage3_spss_materialized_columns.csv', index=False)

block_audit=pd.DataFrame(SPSS_BLOCK_AUDIT)
block_audit.to_csv(LOG_DIR/'stage3_spss_block_lineage.csv',index=False)
block_audit[['block','bytes_processed','slot_millis','run_utc']].to_csv(
    LOG_DIR/'stage3_etl_jobs_metrics.csv',index=False)

lineage=pd.DataFrame([{
    'output_table':A,
    'source_table':A,
    'operation':'replace only SPSS-derived columns; preserve all other columns',
    'authority':'listed SPSS syntax files',
    'block_count':len(block_audit),
    'combined_sql_sha256':hashlib.sha256(''.join(block_audit.sql_sha256).encode()).hexdigest(),
    'run_utc':RUN_UTC}])
lineage.to_csv(LOG_DIR/'stage3_lineage.csv',index=False)

execution_log=(
    '# Stage 03 execution log — CRS04\n\n'
    f'- UTC: {RUN_UTC}\n- Table: `{A}`\n- Rows: {EXPECTED_ROWS}\n'
    f'- SPSS blocks: {len(block_audit)}\n- Derived columns: {len(SPSS_MATERIALIZED)}\n'
    '- Result: PASS 08\n')
(LOG_DIR/'stage3_execution_log.md').write_text(execution_log,encoding='utf-8')
print('PASS 08 | columnas reconstruidas desde SPSS:', len(SPSS_MATERIALIZED))


,filas,conglomerados,errores_hogar,errores_opinion,errores_riesgo
0,18807,1115,0,0,0


PASS 08 | columnas reconstruidas desde SPSS: 730
